# 5. Structured Output

This notebook demonstrates how to get structured output (like JSON or specific schemas) from the language model.

In [9]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")

In [15]:
from langchain.chat_models import init_chat_model

model = init_chat_model(
    "google_genai:gemini-1.5-flash-lite",
)

model

ChatGoogleGenerativeAI(metadata={'lc_versions': {'langchain-core': '1.4.9', 'langchain': '1.3.14', 'langchain-google-genai': '4.2.7'}}, output_version=None, profile={}, google_api_key=SecretStr('**********'), location=None, model='gemini-1.5-flash-lite', temperature=1.0, client=<google.genai.client.Client object at 0x0000021652333A90>, default_metadata=(), model_kwargs={})

## Pyantic

In [16]:
from pydantic import BaseModel , Field


In [17]:
class Movie(BaseModel):
    title: str=Field(description="The title of the movie")
    year: int = Field(description="The year the movie was released")
    director: str = Field(description="The director of the movie")
    rating :float = Field(description="The rating of the movie out of 10")

    

In [18]:
model_with_structure = model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGoogleGenerativeAI(metadata={'lc_versions': {'langchain-core': '1.4.9', 'langchain': '1.3.14', 'langchain-google-genai': '4.2.7'}}, output_version=None, profile={}, google_api_key=SecretStr('**********'), location=None, model='gemini-1.5-flash-lite', temperature=1.0, client=<google.genai.client.Client object at 0x0000021652333A90>, default_metadata=(), model_kwargs={}), kwargs={'response_mime_type': 'application/json', 'response_json_schema': {'properties': {'title': {'description': 'The title of the movie', 'title': 'Title', 'type': 'string'}, 'year': {'description': 'The year the movie was released', 'title': 'Year', 'type': 'integer'}, 'director': {'description': 'The director of the movie', 'title': 'Director', 'type': 'string'}, 'rating': {'description': 'The rating of the movie out of 10', 'title': 'Rating', 'type': 'number'}}, 'required': ['title', 'year', 'director', 'rating'], 'title': 'Movie', 'type': 'object'}, 'ls_structured_output_format': {'kwa

In [19]:
model_with_structure.invoke('provide details of the movie "Inception" ')

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

In [20]:
model.invoke("provide details of the movie 'Inception' ")

AIMessage(content=[{'type': 'text', 'text': '**Inception** is a 2010 science fiction action film written and directed by Christopher Nolan. It stars Leonardo DiCaprio as a professional thief who steals information by infiltrating his targets\' subconscious minds. \n\nHere are the details of the movie, broken down by plot, cast, themes, and legacy:\n\n---\n\n### **1. The Premise & Core Concept**\nThe movie is set in a world where shared dreaming technology exists, allowing people to enter others\' minds. \n* **Extraction:** The main characters are "extractors" who infiltrate a target\'s dreams to steal corporate secrets.\n* **Inception:** The central conflict of the movie is the opposite of extraction—instead of *stealing* an idea, the team must *plant* a new idea into a target’s subconscious so that he believes it is his own.\n\n---\n\n### **2. Plot Summary (No Spoilers for the Ending)**\nDom Cobb (Leonardo DiCaprio) is a skilled fugitive who cannot return to his children in the United

## Message output alogside parsed strcture

In [21]:

class Movie(BaseModel):
    """A movie with details."""
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="The year the movie was released")
    director: str = Field(..., description="The director of the movie")
    rating: float = Field(..., description="The movie's rating out of 10")

model_with_structure = model.with_structured_output(Movie, include_raw=True)  

response = model_with_structure.invoke("Provide details about the movie Inception")
response

{'raw': AIMessage(content=[{'type': 'text', 'text': '{\n  "title": "Inception",\n  "year": 2010,\n  "director": "Christopher Nolan",\n  "rating": 8.8\n}', 'extras': {'signature': 'EjQKMgERTTIPx9P0q1gN9ukY80WvIyI6aoBnsW47wyxdlSEVw3CDO8vYY5zZ0yAoUlLQhZNT'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-1.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019f89b5-c00d-7e00-bfe1-86d27d72ca9c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 8, 'output_tokens': 41, 'total_tokens': 49, 'input_token_details': {'cache_read': 0}}),
 'parsed': Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8),
 'parsing_error': None}

## Nested Structure

In [22]:
class Actor(BaseModel):
    name:str
    role:str

class MovieDetails(BaseModel):
    title: str
    year: int
    director: str
    rating: float
    actors: list[Actor]
    budget: float | None = Field(default=None, description="The budget of the movie in USD")

model_with_structure = model.with_structured_output(MovieDetails, include_raw=True)

response = model_with_structure.invoke("Provide details about the movie Inception, including actors and budget")
response

{'raw': AIMessage(content=[{'type': 'text', 'text': '{\n  "title": "Inception",\n  "year": 2010,\n  "director": "Christopher Nolan",\n  "rating": 8.8,\n  "actors": [\n    {\n      "name": "Leonardo DiCaprio",\n      "role": "Cobb"\n    },\n    {\n      "name": "Joseph Gordon-Levitt",\n      "role": "Arthur"\n    },\n    {\n      "name": "Elliot Page",\n      "role": "Ariadne"\n    },\n    {\n      "name": "Tom Hardy",\n      "role": "Eames"\n    }\n  ],\n  "budget": 160000000\n}', 'extras': {'signature': 'EjQKMgERTTIPUhYm8BO5COG0rkbEyC974NfowpmJ8ePOgPh1QXvL8+KDQEZl8xtdQa2mlpJi'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-1.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019f89bb-936a-7c13-8da1-d7d98d21118a-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 13, 'output_tokens': 166, 'total_tokens': 179, 'input_token_details': {'cache_read': 0}}),
 'parsed': MovieDetails(title='Ince

## TypedDict

In [ ]:
from typing_extensions import Annotated, TypedDict  

class Movie(TypedDict):
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The rating of the movie out of 10"]

In [31]:
model_withtypedict = model.with_structured_output(Movie)

In [32]:
res = model_withtypedict.invoke("Provide details about the movie Inception")

In [33]:
res

{'title': 'Inception',
 'year': 2010,
 'director': 'Christopher Nolan',
 'rating': 8.8}

In [34]:

from pydantic import BaseModel, Field
from langchain.agents import create_agent


class ContactInfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent(
    model="google_genai:gemini-1.5-flash-lite",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result

{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='e285c25f-7c41-4086-9ea9-36ae4e95ec5a'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'ContactInfo', 'arguments': '{"name": "John Doe", "email": "john@example.com", "phone": "(555) 123-4567"}'}, '__gemini_function_call_thought_signatures__': {'gegv515p': 'EjQKMgERTTIPDM/sen3RpHVByJKlZV2RzJAGxPwyhridxbrTZJS6PPEoDiOe9qLGwVAigAFy'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-1.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019f89c6-6d96-7250-82c7-b39d6f114165-0', tool_calls=[{'name': 'ContactInfo', 'args': {'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}, 'id': 'gegv515p', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 127, 'output_tokens': 45, 'total_tokens': 172, 'input_token_details': {'cache